# Diagnóstico de calidad de datos — Pokémon por generaciones

**Autor:** Brandon Maldonado Mesa · Ficha 3169901
**Actividad:** GA1-220501094-01-AA1-EV05

## Momento 1. Reconocimiento del proyecto y del dataset

- **Proyecto:** evolución de las estadísticas de combate de los Pokémon a lo largo de las generaciones.
- **Pregunta analítica:** ¿cómo han cambiado las estadísticas base promedio (HP, ataque, defensa, ataque especial, defensa especial y velocidad) entre generaciones, y cuál de ellas ha tenido el mayor cambio?
- **Dataset:** `data/raw/pokemon_pokeapi.csv`, generado desde PokéAPI con `scripts/descargar_pokeapi.py`. Contiene un registro por Pokémon (generaciones 1 a 9) con id, nombre, generación, tipos, indicadores de legendario y mítico, las seis estadísticas base y el total.
- **Aspectos de calidad a revisar:** valores faltantes (especialmente `tipo_2`), tipos de datos, duplicados, rangos de las estadísticas, coherencia del total con la suma de las seis estadísticas y distribución de Pokémon por generación.

## Momento 2. Carga de datos

In [1]:
import pandas as pd
from pathlib import Path

RUTA = Path("../data/raw/pokemon_pokeapi.csv")

if not RUTA.exists():
    raise FileNotFoundError(
        f"No se encontró {RUTA}. Ejecuta desde la raíz del repo: python scripts\\descargar_pokeapi.py"
    )

df_pokemon = pd.read_csv(RUTA)
print("Archivo cargado:", RUTA)
print("Dimensiones:", df_pokemon.shape)

Archivo cargado: ..\data\raw\pokemon_pokeapi.csv
Dimensiones: (1025, 14)


## Momento 3. Inspección de la estructura

### `head()`

In [2]:
df_pokemon.head()

,id,nombre,generacion,tipo_1,tipo_2,legendario,mitico,hp,ataque,defensa,ataque_especial,defensa_especial,velocidad,total
0,1,bulbasaur,1,grass,poison,False,False,45,49,49,65,65,45,318
1,2,ivysaur,1,grass,poison,False,False,60,62,63,80,80,60,405
2,3,venusaur,1,grass,poison,False,False,80,82,83,100,100,80,525
3,4,charmander,1,fire,NaN,False,False,39,52,43,60,50,65,309
4,5,charmeleon,1,fire,NaN,False,False,58,64,58,80,65,80,405


### `tail()`

In [3]:
df_pokemon.tail()

,id,nombre,generacion,tipo_1,tipo_2,legendario,mitico,hp,ataque,defensa,ataque_especial,defensa_especial,velocidad,total
1020,1021,raging-bolt,9,electric,dragon,False,False,125,73,91,137,89,75,590
1021,1022,iron-boulder,9,rock,psychic,False,False,90,120,80,68,108,124,590
1022,1023,iron-crown,9,steel,psychic,False,False,90,72,100,122,108,98,590
1023,1024,terapagos,9,normal,NaN,True,False,90,65,85,65,85,60,450
1024,1025,pecharunt,9,poison,ghost,False,True,88,88,160,88,88,88,600


### `shape`

In [4]:
print("Filas:", df_pokemon.shape[0])
print("Columnas:", df_pokemon.shape[1])
df_pokemon.shape

Filas: 1025
Columnas: 14


(1025, 14)

### `columns`

In [5]:
df_pokemon.columns

Index(['id', 'nombre', 'generacion', 'tipo_1', 'tipo_2', 'legendario',
       'mitico', 'hp', 'ataque', 'defensa', 'ataque_especial',
       'defensa_especial', 'velocidad', 'total'],
      dtype='str')

### `dtypes`

In [6]:
df_pokemon.dtypes

id                  int64
nombre                str
generacion          int64
tipo_1                str
tipo_2                str
legendario           bool
mitico               bool
hp                  int64
ataque              int64
defensa             int64
ataque_especial     int64
defensa_especial    int64
velocidad           int64
total               int64
dtype: object

### `info()`

In [7]:
df_pokemon.info()

<class 'pandas.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id                1025 non-null   int64
 1   nombre            1025 non-null   str  
 2   generacion        1025 non-null   int64
 3   tipo_1            1025 non-null   str  
 4   tipo_2            526 non-null    str  
 5   legendario        1025 non-null   bool 
 6   mitico            1025 non-null   bool 
 7   hp                1025 non-null   int64
 8   ataque            1025 non-null   int64
 9   defensa           1025 non-null   int64
 10  ataque_especial   1025 non-null   int64
 11  defensa_especial  1025 non-null   int64
 12  velocidad         1025 non-null   int64
 13  total             1025 non-null   int64
dtypes: bool(2), int64(9), str(3)
memory usage: 98.2 KB


### `describe()`

In [8]:
df_pokemon.describe()

,id,generacion,hp,ataque,defensa,ataque_especial,defensa_especial,velocidad,total
count,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000
mean,513.000000,4.741463,70.184390,77.521951,72.507317,70.080976,70.205854,67.186341,427.686829
std,296.036315,2.633633,26.631054,29.782541,29.286972,29.658378,26.639329,28.717227,112.770735
min,1.000000,1.000000,1.000000,5.000000,5.000000,10.000000,20.000000,5.000000,175.000000
25%,257.000000,3.000000,50.000000,55.000000,50.000000,47.000000,50.000000,45.000000,325.000000
50%,513.000000,5.000000,68.000000,75.000000,70.000000,65.000000,67.000000,65.000000,450.000000
75%,769.000000,7.000000,85.000000,100.000000,90.000000,90.000000,86.000000,88.000000,508.000000
max,1025.000000,9.000000,255.000000,181.000000,230.000000,173.000000,230.000000,200.000000,720.000000


### `describe(include='object')`

Se usa la forma pedida en la guía. Con versiones recientes de Pandas, las columnas de texto pueden venir con tipo `str` en lugar de `object`; si la primera forma no devuelve nada, se usa una alternativa equivalente.

In [9]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    try:
        resumen_texto = df_pokemon.describe(include='object')
    except ValueError:
        resumen_texto = pd.DataFrame()

if resumen_texto.empty:
    resumen_texto = df_pokemon.describe(exclude='number')

resumen_texto

,nombre,tipo_1,tipo_2
count,1025,1025,526
unique,1025,18,18
top,bulbasaur,water,flying
freq,1,134,100


### Tabla de diagnóstico estructural

In [10]:
tabla_estructural = pd.DataFrame([{
    "Dataset": "pokemon_pokeapi.csv",
    "Filas": df_pokemon.shape[0],
    "Columnas": df_pokemon.shape[1],
    "Tipos de datos": ", ".join(f"{t} ({n})" for t, n in df_pokemon.dtypes.astype(str).value_counts().items()),
    "Observaciones": "Un registro por Pokémon, sin duplicados. Solo tipo_2 tiene nulos (499). Tipos de dato coherentes con cada variable.",
}])
tabla_estructural

,Dataset,Filas,Columnas,Tipos de datos,Observaciones
0,pokemon_pokeapi.csv,1025,14,"int64 (9), str (3), bool (2)",Completar según los resultados anteriores


### Características principales del dataset

### Características principales del dataset

- 1025 registros y 14 columnas: 9 enteras, 3 de texto y 2 booleanas.
- Las seis estadísticas van de 1 a 255 (hp), 5 a 181 (ataque), 5 a 230 (defensa), 10 a 173 (ataque especial), 20 a 230 (defensa especial) y 5 a 200 (velocidad).
- Solo `tipo_2` tiene valores nulos: 499, que corresponden a Pokémon de un solo tipo.
- El total promedio es 427.7 y va de 175 a 720.

## Momento 4. Diagnóstico inicial de calidad

### 4.1 Valores faltantes

In [11]:
nulos = df_pokemon.isna().sum()
tabla_nulos = pd.DataFrame({
    "nulos": nulos,
    "porcentaje": (nulos / len(df_pokemon) * 100).round(2),
})
print("Columnas con valores faltantes:")
tabla_nulos[tabla_nulos["nulos"] > 0]

Columnas con valores faltantes:


,nulos,porcentaje
tipo_2,499,48.68


In [12]:
# Cadenas vacías o solo con espacios en columnas de texto
columnas_texto = df_pokemon.select_dtypes(exclude=["number", "bool"]).columns
vacias = {c: int((df_pokemon[c].dropna().astype(str).str.strip() == "").sum()) for c in columnas_texto}
vacias

{'nombre': 0, 'tipo_1': 0, 'tipo_2': 0}

In [13]:
# ¿Los nulos de tipo_2 corresponden a Pokémon de un solo tipo?
print("Pokémon con un solo tipo (tipo_2 nulo):", df_pokemon["tipo_2"].isna().sum())
print("Pokémon con dos tipos:", df_pokemon["tipo_2"].notna().sum())

Pokémon con un solo tipo (tipo_2 nulo): 499
Pokémon con dos tipos: 526


### 4.2 Tipos de datos

In [14]:
esperados = {
    "id": "entero", "nombre": "texto", "generacion": "entero",
    "tipo_1": "texto", "tipo_2": "texto",
    "legendario": "booleano", "mitico": "booleano",
    "hp": "entero", "ataque": "entero", "defensa": "entero",
    "ataque_especial": "entero", "defensa_especial": "entero",
    "velocidad": "entero", "total": "entero",
}

def categoria(serie):
    if pd.api.types.is_bool_dtype(serie):
        return "booleano"
    if pd.api.types.is_integer_dtype(serie):
        return "entero"
    if pd.api.types.is_float_dtype(serie):
        return "decimal"
    return "texto"

tabla_tipos = pd.DataFrame({
    "dtype_actual": df_pokemon.dtypes.astype(str),
    "categoria_actual": [categoria(df_pokemon[c]) for c in df_pokemon.columns],
})
tabla_tipos["tipo_esperado"] = [esperados.get(c, "sin definir") for c in tabla_tipos.index]
tabla_tipos["coherente"] = tabla_tipos["categoria_actual"] == tabla_tipos["tipo_esperado"]
tabla_tipos

,dtype_actual,categoria_actual,tipo_esperado,coherente
id,int64,entero,entero,True
nombre,str,texto,texto,True
generacion,int64,entero,entero,True
tipo_1,str,texto,texto,True
tipo_2,str,texto,texto,True
legendario,bool,booleano,booleano,True
mitico,bool,booleano,booleano,True
hp,int64,entero,entero,True
ataque,int64,entero,entero,True
defensa,int64,entero,entero,True


### 4.3 Duplicados

In [15]:
print("Filas completamente duplicadas:", df_pokemon.duplicated().sum())
print("IDs repetidos:", df_pokemon["id"].duplicated().sum())
print("Nombres repetidos:", df_pokemon["nombre"].duplicated().sum())

Filas completamente duplicadas: 0
IDs repetidos: 0
Nombres repetidos: 0


### 4.4 Posibles inconsistencias

In [16]:
estadisticas = ["hp", "ataque", "defensa", "ataque_especial", "defensa_especial", "velocidad"]
tipos_validos = {
    "normal", "fire", "water", "electric", "grass", "ice", "fighting", "poison",
    "ground", "flying", "psychic", "bug", "rock", "ghost", "dragon", "dark",
    "steel", "fairy",
}

# 1) El total debe ser la suma de las seis estadísticas
suma = df_pokemon[estadisticas].sum(axis=1)
total_incoherente = df_pokemon[df_pokemon["total"] != suma]

# 2) Estadísticas fuera de un rango razonable (0 a 255 es el máximo posible en los juegos)
fuera_rango = df_pokemon[(df_pokemon[estadisticas] <= 0).any(axis=1) | (df_pokemon[estadisticas] > 255).any(axis=1)]

# 3) Generación fuera de 1-9
gen_invalida = df_pokemon[~df_pokemon["generacion"].between(1, 9)]

# 4) El id debe ser consecutivo (1..N) y la generación no debe bajar al subir el id
ids_ordenados = df_pokemon["id"].sort_values().reset_index(drop=True)
id_consecutivo = bool((ids_ordenados == pd.Series(range(1, len(df_pokemon) + 1))).all())
generacion_monotona = df_pokemon.sort_values("id")["generacion"].is_monotonic_increasing

# 5) Tipos que no existen
t1_invalido = df_pokemon[~df_pokemon["tipo_1"].isin(tipos_validos)]
t2_invalido = df_pokemon[df_pokemon["tipo_2"].notna() & ~df_pokemon["tipo_2"].isin(tipos_validos)]

# 6) tipo_2 igual a tipo_1
tipos_iguales = df_pokemon[df_pokemon["tipo_1"] == df_pokemon["tipo_2"]]

# 7) Legendario y mítico a la vez
leg_y_mit = df_pokemon[df_pokemon["legendario"] & df_pokemon["mitico"]]

verificaciones = pd.DataFrame([
    ("Total distinto de la suma de las 6 estadísticas", len(total_incoherente)),
    ("Estadísticas <= 0 o > 255", len(fuera_rango)),
    ("Generación fuera de 1-9", len(gen_invalida)),
    ("IDs no consecutivos (1 = hay problema)", int(not id_consecutivo)),
    ("Generación que baja al subir el id (1 = hay problema)", int(not generacion_monotona)),
    ("tipo_1 inválido", len(t1_invalido)),
    ("tipo_2 inválido", len(t2_invalido)),
    ("tipo_2 igual a tipo_1", len(tipos_iguales)),
    ("Legendario y mítico a la vez", len(leg_y_mit)),
], columns=["Verificación", "Casos encontrados"])
verificaciones

,Verificación,Casos encontrados
0,Total distinto de la suma de las 6 estadísticas,0
1,Estadísticas <= 0 o > 255,0
2,Generación fuera de 1-9,0
3,IDs no consecutivos (1 = hay problema),0
4,Generación que baja al subir el id (1 = hay pr...,0
5,tipo_1 inválido,0
6,tipo_2 inválido,0
7,tipo_2 igual a tipo_1,0
8,Legendario y mítico a la vez,0


### 4.5 Limitaciones frente a la pregunta analítica

In [17]:
# Pokémon por generación frente a la referencia oficial de cada generación
referencia = {1: 151, 2: 100, 3: 135, 4: 107, 5: 156, 6: 72, 7: 88, 8: 96, 9: 120}
por_gen = df_pokemon.groupby("generacion").agg(
    pokemon=("id", "count"),
    legendarios=("legendario", "sum"),
    miticos=("mitico", "sum"),
    total_promedio=("total", "mean"),
).round(1)
por_gen["referencia"] = por_gen.index.map(referencia)
por_gen["diferencia"] = por_gen["pokemon"] - por_gen["referencia"]
por_gen

,pokemon,legendarios,miticos,total_promedio,referencia,diferencia
generacion,,,,,,
1,151,4,1,407.6,151,0
2,100,5,1,407.2,100,0
3,135,8,2,403.7,135,0
4,107,9,5,445.6,107,0
5,156,9,4,425.8,156,0
6,72,3,3,429.3,72,0
7,88,11,5,449.4,88,0
8,96,11,1,439.2,96,0
9,120,11,1,457.4,120,0


In [18]:
# Valores atípicos por estadística (regla del rango intercuartílico)
atipicos = {}
for c in estadisticas + ["total"]:
    q1, q3 = df_pokemon[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    mascara = (df_pokemon[c] < q1 - 1.5 * iqr) | (df_pokemon[c] > q3 + 1.5 * iqr)
    atipicos[c] = int(mascara.sum())
pd.Series(atipicos, name="valores atípicos")

hp                  18
ataque               1
defensa             11
ataque_especial      1
defensa_especial    11
velocidad            2
total                0
Name: valores atípicos, dtype: int64

In [19]:
# Efecto de los legendarios y míticos sobre el promedio del total
especiales = df_pokemon["legendario"] | df_pokemon["mitico"]
print("Promedio del total, todos:", round(df_pokemon["total"].mean(), 1))
print("Promedio del total, sin legendarios ni míticos:", round(df_pokemon.loc[~especiales, "total"].mean(), 1))
print("Promedio del total, solo legendarios y míticos:", round(df_pokemon.loc[especiales, "total"].mean(), 1))

Promedio del total, todos: 427.7
Promedio del total, sin legendarios ni míticos: 411.1
Promedio del total, solo legendarios y míticos: 592.0


### 4.6 Tabla de problemas identificados

| Dataset | Problema identificado | Evidencia encontrada | Posible impacto | Acción propuesta |
|---|---|---|---|---|
| pokemon_pokeapi.csv | `tipo_2` con muchos nulos | 499 nulos (48.68 %), equivalen a Pokémon de un solo tipo | Afecta solo los análisis por tipo secundario | Conservar los nulos; usar la categoría `ninguno` solo al analizar por tipo |
| pokemon_pokeapi.csv | Valores atípicos en las estadísticas | 18 en hp, 11 en defensa, 11 en defensa especial, 2 en velocidad | Pueden desplazar los promedios por generación | Listar los registros y confirmar su validez con una segunda fuente |
| pokemon_pokeapi.csv | Legendarios y míticos sesgan el promedio | Total promedio 592.0 frente a 411.1 del resto | Generaciones con más legendarios podrían parecer más fuertes | Calcular promedios con y sin estos casos |
| pokemon_pokeapi.csv | Una sola fuente y una sola forma por Pokémon | 1025 registros, sin formas alternativas | No permite contrastar valores ni estudiar megaevoluciones o formas regionales | Contrastar una muestra con otra fuente y definir el alcance |